In [ ]:
import os
import re
import glob
from dataclasses import dataclass
from typing import List, Dict, Tuple

import numpy as np

# PDF text extraction
import pdfplumber

# Similarity + clustering
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ----------------------------
# Config you can tweak
# ----------------------------
DEFAULT_SIM_THRESHOLD = 0.78   # higher = stricter grouping (try 0.75–0.88)
MAX_PAGES_TO_SAMPLE = 1        # templates usually show up on page 1
MAX_LINES = 80                 # take only first N lines (stable header/table labels)
DROP_NUMBER_HEAVY_LINES = True # drop lines that are mostly numbers (NAV values etc.)


@dataclass
class PdfTemplateFingerprint:
    path: str
    fingerprint_text: str


def _normalize_line(s: str) -> str:
    s = s.strip()
    s = re.sub(r"\s+", " ", s)

    # Remove things that vary per statement
    s = re.sub(r"\b\d{1,2}[-/]\d{1,2}[-/]\d{2,4}\b", " <DATE> ", s)  # dates like 01-02-2026
    s = re.sub(r"\b\d{4}[-/]\d{1,2}[-/]\d{1,2}\b", " <DATE> ", s)    # dates like 2026-03-02
    s = re.sub(r"\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b", " <EMAIL> ", s, flags=re.I)

    # Replace long numbers / currency-like values (NAV, amounts, folio, etc.)
    s = re.sub(r"₹", " INR ", s)
    s = re.sub(r"\bINR\b", " INR ", s, flags=re.I)
    s = re.sub(r"\b\d{6,}\b", " <LONGNUM> ", s)                # long ids
    s = re.sub(r"\b\d+\.\d+\b", " <DECIMAL> ", s)              # decimals
    s = re.sub(r"\b\d+\b", " <NUM> ", s)                       # any remaining integers

    # Keep letters and common punctuation for template cues
    s = re.sub(r"[^a-zA-Z<>/ _:-]", " ", s)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s


def _is_number_heavy(line: str) -> bool:
    # If line was mostly numbers before normalization, it won't help template grouping
    raw = re.sub(r"\s+", "", line)
    if not raw:
        return True
    digits = sum(ch.isdigit() for ch in raw)
    return digits / max(1, len(raw)) > 0.45


def fingerprint_pdf(pdf_path: str,
                    max_pages: int = MAX_PAGES_TO_SAMPLE,
                    max_lines: int = MAX_LINES) -> PdfTemplateFingerprint:
    """
    Create a template fingerprint string for a PDF:
    - Extract page text (usually page 1 is enough)
    - Keep early lines (header, column labels, section titles)
    - Normalize away dates/numbers (variable content)
    """
    lines_out: List[str] = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            pages = pdf.pages[:max_pages]
            for p in pages:
                txt = p.extract_text() or ""
                # Split into lines; keep only first chunk (template markers)
                raw_lines = txt.splitlines()[:max_lines]
                for ln in raw_lines:
                    if DROP_NUMBER_HEAVY_LINES and _is_number_heavy(ln):
                        continue
                    norm = _normalize_line(ln)
                    if norm and len(norm) >= 3:
                        lines_out.append(norm)

        # De-dup while preserving order (stable)
        seen = set()
        uniq = []
        for l in lines_out:
            if l not in seen:
                uniq.append(l)
                seen.add(l)

        fp_text = "\n".join(uniq)
        return PdfTemplateFingerprint(path=pdf_path, fingerprint_text=fp_text)

    except Exception as e:
        # If a PDF is corrupted / scanned-only, you may get empty text
        return PdfTemplateFingerprint(path=pdf_path, fingerprint_text=f"__ERROR__ {e}")


def cluster_by_similarity(fps: List[PdfTemplateFingerprint],
                          sim_threshold: float = DEFAULT_SIM_THRESHOLD
                          ) -> Tuple[List[int], np.ndarray]:
    """
    Clustering via similarity graph:
    - Compute cosine similarity matrix
    - Make edges where sim >= threshold
    - Connected components = clusters
    """
    docs = [fp.fingerprint_text for fp in fps]
    vec = TfidfVectorizer(min_df=1, ngram_range=(1, 2))
    X = vec.fit_transform(docs)

    sim = cosine_similarity(X)
    n = sim.shape[0]

    # Build adjacency list
    adj: Dict[int, List[int]] = {i: [] for i in range(n)}
    for i in range(n):
        for j in range(i + 1, n):
            if sim[i, j] >= sim_threshold:
                adj[i].append(j)
                adj[j].append(i)

    # Connected components
    labels = [-1] * n
    cluster_id = 0
    for i in range(n):
        if labels[i] != -1:
            continue
        stack = [i]
        labels[i] = cluster_id
        while stack:
            u = stack.pop()
            for v in adj[u]:
                if labels[v] == -1:
                    labels[v] = cluster_id
                    stack.append(v)
        cluster_id += 1

    return labels, sim


def classify_folder(pdf_folder: str,
                    pattern: str = "*.pdf",
                    sim_threshold: float = DEFAULT_SIM_THRESHOLD) -> Dict[int, List[str]]:
    pdf_paths = sorted(glob.glob(os.path.join(pdf_folder, pattern)))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs found in: {pdf_folder}")

    fps = [fingerprint_pdf(p) for p in pdf_paths]
    labels, sim = cluster_by_similarity(fps, sim_threshold=sim_threshold)

    groups: Dict[int, List[str]] = {}
    for fp, lab in zip(fps, labels):
        groups.setdefault(lab, []).append(fp.path)

    # Print a quick summary
    print(f"Total PDFs: {len(pdf_paths)}")
    print(f"Clusters found: {len(groups)} (threshold={sim_threshold})")
    for k in sorted(groups):
        print(f"\nCluster {k} ({len(groups[k])} files):")
        for p in groups[k]:
            print("  -", os.path.basename(p))

    return groups


if __name__ == "__main__":
    # Example usage:
    # classify_folder(r"C:\data\cas_pdfs", sim_threshold=0.80)
    folder = os.environ.get("CAS_PDF_FOLDER", "")
    if folder:
        classify_folder(folder, sim_threshold=DEFAULT_SIM_THRESHOLD)
    else:
        print("Set CAS_PDF_FOLDER env var or call classify_folder(...) directly.")